In [1]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import os
import random
import re
import time

import pandas as pd
import requests
from tqdm import tqdm



In [2]:
BASE_URL = "https://www.herbchambers.com"
DATA_DIR = "./data"

HEADERS = {
    "User-Agent": os.getenv(
        "HERBCHAMBERS_USER_AGENT",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": os.getenv("HERBCHAMBERS_ACCEPT_LANGUAGE", "en-US,en;q=0.9"),
    "Accept-Encoding": "gzip, deflate",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1",
}



In [3]:
def request_headers(referer=None):
    headers = dict(HEADERS)
    cookie = os.getenv("HERBCHAMBERS_COOKIE")
    if cookie:
        headers["Cookie"] = cookie
    if referer:
        headers["Referer"] = referer
    return headers


def sleep_for_delay(delay_seconds):
    if isinstance(delay_seconds, tuple):
        low, high = delay_seconds
        time.sleep(random.uniform(min(low, high), max(low, high)))
    else:
        time.sleep(delay_seconds)


def get_soup(url, delay_seconds=0.75, referer=BASE_URL):
    sleep_for_delay(delay_seconds)
    response = requests.get(url, headers=request_headers(referer=referer), timeout=30)
    if response.status_code == 403:
        print(f"Blocked with HTTP 403 for {url}")
        print("Use a legitimate HERBCHAMBERS_COOKIE / HERBCHAMBERS_USER_AGENT from your browser session, reduce volume, or use a permitted data source.")
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def clean_text(value):
    if not value:
        return None
    return re.sub(r"\s+", " ", value).strip()


def parse_money(value):
    if not value:
        return None
    match = re.search(r"[\d,]+(?:\.\d+)?", str(value))
    return float(match.group(0).replace(",", "")) if match else None


def parse_integer(value):
    if not value:
        return None
    match = re.search(r"[\d,]+", str(value))
    return int(match.group(0).replace(",", "")) if match else None


def slug_label(value):
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")



In [4]:
def split_title(title):
    if not title:
        return [None, None, None, None]
    clean = re.sub(r"^(Used|New|Certified)\s+", "", title.strip(), flags=re.IGNORECASE)
    parts = clean.split()
    if len(parts) < 3 or not parts[0].isdigit():
        return [None, None, None, None]
    year = int(parts[0])
    make = parts[1]
    model = parts[2]
    trim = " ".join(parts[3:]) if len(parts) > 3 else None
    return [year, make, model, trim]


def listing_card_containers(soup):
    return soup.select(".vehicle-card-details-container")


def text_from_class(card, class_name):
    element = card.select_one(f".{class_name}")
    return clean_text(element.get_text(" ", strip=True)) if element else None


def first_price(card, selector):
    element = card.select_one(selector)
    return parse_money(element.get_text(" ", strip=True)) if element else None


def price_description(card):
    pieces = []
    for label in card.select("dl.pricing-detail dt"):
        value = label.find_next_sibling("dd")
        label_text = clean_text(label.get_text(" ", strip=True))
        value_text = clean_text(value.get_text(" ", strip=True)) if value else None
        if label_text and value_text:
            pieces.append(f"{label_text}: {value_text}")
    return "; ".join(pieces) or None


def parse_stock(value):
    if not value:
        return None
    match = re.search(r"Stock\s*#?\s*([A-Za-z0-9-]+)", value)
    return match.group(1) if match else value



In [5]:
def parse_inventory_card(card, inventory_url, source_name, inventory_kind):
    title_link = card.select_one(".vehicle-card-title a[href]")
    if not title_link:
        return None

    title = clean_text(title_link.get_text(" ", strip=True))
    detail_url = urljoin(BASE_URL, title_link.get("href", ""))
    question_link = card.select_one("[data-vin]")
    year, make, model, trim = split_title(title)
    if question_link:
        year = parse_integer(question_link.get("data-year")) or year
        make = question_link.get("data-make") or make
        model = question_link.get("data-model") or model
        trim = question_link.get("data-trim") or trim

    exterior = text_from_class(card, "exteriorColor")
    interior = text_from_class(card, "interiorColor")
    if exterior:
        exterior = exterior.replace(" Exterior", "").strip()
    if interior:
        interior = interior.replace(" Interior", "").strip()

    dealer = card.select_one(".accountName span[aria-hidden='true']")
    dealer_info = clean_text(dealer.get_text(" ", strip=True)) if dealer else None
    image = card.find("img", src=re.compile(r"vehicle|inventory|photos|pictures", re.IGNORECASE))
    listing_image_url = urljoin(BASE_URL, image["src"]) if image and image.get("src") else None

    row = {
        "url": detail_url,
        "vin": question_link.get("data-vin") if question_link else None,
        "title": title,
        "year": year,
        "make": make,
        "model": model,
        "trim": trim,
        "sales_price": first_price(card, ".salePrice .price-value") or first_price(card, ".askingPrice .price-value"),
        "odometer_miles": parse_integer(text_from_class(card, "highlight-badge")),
        "exterior": exterior,
        "interior": interior,
        "dealer_info": dealer_info,
        "source_name": source_name,
        "inventory_kind": inventory_kind,
        "stock_number": question_link.get("data-stock") if question_link else parse_stock(text_from_class(card, "stockNumber")),
        "listing_image_url": listing_image_url,
        "price_description": price_description(card),
        "drivetrain": text_from_class(card, "driveLine"),
        "engine": text_from_class(card, "engine"),
        "return_to": inventory_url,
    }
    return row



In [6]:
def inventory_page_url(base_url, page):
    if page <= 1:
        return base_url
    separator = "&" if "?" in base_url else "?"
    return f"{base_url}{separator}start={(page - 1) * 18}"


def parse_cards_from_soup(soup, page_url, source_name, inventory_kind):
    rows = []
    cards = listing_card_containers(soup)
    if not cards:
        print("No listing cards found in the supplied page.")
        return pd.DataFrame()

    for card in cards:
        row = parse_inventory_card(card, page_url, source_name, inventory_kind)
        if row:
            rows.append(row)

    df = pd.DataFrame(rows)
    if "url" in df.columns:
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    return df


def collect_inventory_cards(inventory_url, source_name, inventory_kind, max_pages=1, delay_seconds=(8, 25)):
    frames = []
    for page in tqdm(range(1, max_pages + 1), desc=f"{source_name} pages"):
        page_url = inventory_page_url(inventory_url, page)
        try:
            soup = get_soup(page_url, delay_seconds=delay_seconds, referer=BASE_URL)
        except requests.RequestException as exc:
            print(f"Request failed for page {page}: {exc}")
            print("Live request stopped. If the page opens in your browser, save the page HTML and set LOCAL_HTML_PATH in the final cell.")
            break

        page_df = parse_cards_from_soup(soup, page_url, source_name, inventory_kind)
        if page_df.empty:
            break
        frames.append(page_df)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["url"]).reset_index(drop=True)


def collect_inventory_cards_from_html(html_path, inventory_url, source_name, inventory_kind):
    html_path = str(html_path).strip()
    if not html_path:
        return pd.DataFrame()
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    return parse_cards_from_soup(soup, inventory_url, source_name, inventory_kind)


def save_inventory_cards(df, output_prefix, data_dir=DATA_DIR):
    os.makedirs(data_dir, exist_ok=True)
    json_path = os.path.join(data_dir, f"{output_prefix}_links.json")
    csv_path = os.path.join(data_dir, f"{output_prefix}_links.csv")
    records = df.to_dict(orient="records")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)
    df.to_csv(csv_path, index=False)
    print(f"Saved {len(df)} listing cards")
    print(f"- {json_path}")
    print(f"- {csv_path}")
    return json_path, csv_path



In [7]:
def extract_car_details(entry, delay_seconds=(3, 10)):
    car_data = entry.copy()
    link = entry["url"]
    try:
        soup = get_soup(link, delay_seconds=delay_seconds, referer=entry.get("return_to") or BASE_URL)
    except requests.RequestException as exc:
        car_data["scrape_error"] = str(exc)
        return car_data

    try:
        seller_notes_header = soup.find("h2", string=re.compile(r"Seller Notes", re.IGNORECASE))
        if seller_notes_header:
            seller_div = seller_notes_header.find_next("div", class_="see-more")
            if seller_div:
                car_data["seller_notes"] = clean_text(seller_div.get_text(" ", strip=True))

        for heading, output_name in [
            ("Options & packages", "options_and_packages"),
            ("Popular features", "popular_features"),
            ("Standard features", "standard_features"),
        ]:
            header = soup.find("h2", string=re.compile(re.escape(heading), re.IGNORECASE))
            if header:
                container = header.find_next("div")
                if container:
                    items = [clean_text(item.get_text(" ", strip=True)) for item in container.find_all("div", class_="flex items-center")]
                    items = [item for item in items if item]
                    if items:
                        car_data[output_name] = "; ".join(dict.fromkeys(items))

        for item in soup.select("dl dt"):
            value = item.find_next_sibling("dd")
            key = clean_text(item.get_text(" ", strip=True))
            val = clean_text(value.get_text(" ", strip=True)) if value else None
            if key and val:
                car_data[f"detail_{slug_label(key)}"] = val
    except Exception as exc:
        car_data["scrape_error"] = str(exc)

    return car_data


def scrape_all_car_details(json_path, output_json, output_csv, delay_seconds=(3, 10)):
    with open(json_path, "r", encoding="utf-8") as f:
        car_links = json.load(f)

    results = []
    for entry in tqdm(car_links, desc="Scraping car details"):
        results.append(extract_car_details(entry, delay_seconds=delay_seconds))

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print("Saved detail data to:")
    print(f"- {output_csv}")
    print(f"- {output_json}")
    return df



In [8]:
INVENTORY_URL = "https://www.herbchambers.com/used-inventory/index.htm?geoZip=02151&geoRadius=0"
SOURCE_NAME = "Herb Chambers Used"
INVENTORY_KIND = "used"
OUTPUT_PREFIX = "herb_chambers_used"

# First trial: one inventory page and no detail-page fan-out yet.
# If live requests are blocked, save the browser page HTML and set LOCAL_HTML_PATH.
MAX_PAGES = 1
SCRAPE_DETAILS = False
LOCAL_HTML_PATH = ""  # Example: r"data/herb_chambers_used_page1.html"
LISTING_DELAY_SECONDS = (8, 25)
DETAIL_DELAY_SECONDS = (3, 10)

if LOCAL_HTML_PATH:
    cards_df = collect_inventory_cards_from_html(
        LOCAL_HTML_PATH,
        INVENTORY_URL,
        source_name=SOURCE_NAME,
        inventory_kind=INVENTORY_KIND,
    )
else:
    cards_df = collect_inventory_cards(
        INVENTORY_URL,
        source_name=SOURCE_NAME,
        inventory_kind=INVENTORY_KIND,
        max_pages=MAX_PAGES,
        delay_seconds=LISTING_DELAY_SECONDS,
    )

links_json, links_csv = save_inventory_cards(cards_df, OUTPUT_PREFIX)

if SCRAPE_DETAILS and not cards_df.empty:
    details_df = scrape_all_car_details(
        links_json,
        output_json=f"{DATA_DIR}/{OUTPUT_PREFIX}_details.json",
        output_csv=f"{DATA_DIR}/{OUTPUT_PREFIX}_details.csv",
        delay_seconds=DETAIL_DELAY_SECONDS,
    )
else:
    details_df = pd.DataFrame()

cards_df.head()



Herb Chambers Used pages:   0%|          | 0/1 [00:23<?, ?it/s]

Blocked with HTTP 403 for https://www.herbchambers.com/used-inventory/index.htm?geoZip=02151&geoRadius=0
Use a legitimate HERBCHAMBERS_COOKIE / HERBCHAMBERS_USER_AGENT from your browser session, reduce volume, or use a permitted data source.
Request failed for page 1: 403 Client Error: Forbidden for url: https://www.herbchambers.com/used-inventory/index.htm?geoZip=02151&geoRadius=0
Live request stopped. If the page opens in your browser, save the page HTML and set LOCAL_HTML_PATH in the final cell.
Saved 0 listing cards
- ./data\herb_chambers_used_links.json
- ./data\herb_chambers_used_links.csv


""
